#### Store predictions and model metrics in RDS Database

In [ ]:

from dotenv import load_dotenv
import os
import mysql.connector

load_dotenv()

#  connect to RDS 
conn = mysql.connector.connect(
    host='database-cw2.c7oc86yis1kb.ap-south-1.rds.amazonaws.com',
    user='admin',
    password=os.getenv("DB_PASSWORD"),
    port=3306
)
cursor = conn.cursor()

# create database
cursor.execute("CREATE DATABASE IF NOT EXISTS database-cw2")
cursor.execute("USE database-cw2")

# create predictions table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS predictions (
        id                     INT AUTO_INCREMENT PRIMARY KEY,
        prediction_date        TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        customer_index         INT,
        churn_probability      FLOAT,
        predicted_churn        INT,
        actual_churn           INT
    )
""")

# create model metrics table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS model_metrics (
        id              INT AUTO_INCREMENT PRIMARY KEY,
        run_date        TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        model_name      VARCHAR(100),
        roc_auc         FLOAT,
        accuracy        FLOAT,
        precision_score FLOAT,
        recall_score    FLOAT,
        f1_score        FLOAT,
        cv_auc_mean     FLOAT,
        cv_auc_std      FLOAT
    )
""")

conn.commit()
print("Database and tables created successfully")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# generate predictions
y_pred = best_model.predict(X_test_s)
y_prob = best_model.predict_proba(X_test_s)[:, 1]

cursor.execute("USE churn_db")

# insert each prediction
insert_prediction = """
    INSERT INTO predictions 
    (customer_index, churn_probability, predicted_churn, actual_churn)
    VALUES (%s, %s, %s, %s)
"""

records = [
    (int(i),
     float(y_prob[i]),
     int(y_pred[i]),
     int(y_test.iloc[i]))
    for i in range(len(y_pred))
]

cursor.executemany(insert_prediction, records)
conn.commit()
print(f"{cursor.rowcount} predictions stored in RDS successfully")

In [ ]:
from sklearn.metrics import roc_auc_score

# calculate metrics
auc       = roc_auc_score(y_test, y_prob)
acc       = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

# cv scores from earlier
cv_mean = results[best_name]['cv_mean']
cv_std  = results[best_name]['cv_std']

insert_metrics = """
    INSERT INTO model_metrics
    (model_name, roc_auc, accuracy, precision_score, 
     recall_score, f1_score, cv_auc_mean, cv_auc_std)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

cursor.execute(insert_metrics, (
    best_name,
    float(auc),
    float(acc),
    float(precision),
    float(recall),
    float(f1),
    float(cv_mean),
    float(cv_std)
))

conn.commit()
print("Model metrics stored in RDS successfully")

In [ ]:
# check predictions table
cursor.execute("SELECT COUNT(*) FROM churn_db.predictions")
print(f"Total predictions stored: {cursor.fetchone()[0]}")

# check metrics table
cursor.execute("SELECT * FROM churn_db.model_metrics")
for row in cursor.fetchall():
    print(row)

# check high risk customers
cursor.execute("""
    SELECT customer_index, churn_probability 
    FROM churn_db.predictions 
    WHERE churn_probability >= 0.35 
    ORDER BY churn_probability DESC 
    LIMIT 10
""")
print("\nTop 10 highest risk customers:")
for row in cursor.fetchall():
    print(f"  Customer {row[0]} — churn probability: {row[1]:.4f}")